# The second model — with how the players have been playing

The project plan asks for two models: one using only what is known before a map starts with
no player statistics, and one that adds them. Notebook 04 built the first. It scored 51.4%
against a baseline of 55.5%, so it lost to a rule that fits in one sentence.

Notebook 05 was a rough check that the second model was worth building at all. It was.
This notebook is the real version, and it differs from the probe in three ways:

- The feature lives in [`src/form.py`](../src/form.py) rather than in notebook cells, so the
  same code produces the numbers everywhere.
- Older maps can count for less than recent ones, which the probe could not do.
- The cut-off is **proved** rather than asserted, by [`test_form.py`](../test_form.py).

## The one thing that could ruin this

Everything here is built from columns describing what happened *during* a map — combat
scores, damage, ratings. Those are exactly the columns the project has spent two days
refusing to use, because using a map's own statistics to predict that map is reading the
answer off the back of the page.

They become legitimate only when turned into a record of **earlier** matches. A player's
average going into a match is knowable before it starts; their performance in it is not.

That makes the cut-off the whole ball game. Off by one match and the result leaks into its
own prediction, the score looks wonderful, and the model is worthless — which is the most
likely explanation for the 93% accuracy reported by one of the projects in
`existing_projects.txt`.

So it is tested rather than trusted. `test_form.py` runs four checks:

1. **Worked by hand.** Take one real map, work both sides' averages out manually from the
   matches before it, and confirm the code returns those numbers.
2. **The future cannot reach backwards.** Rebuild the features from a truncated season and
   confirm nothing computed earlier moved. If a later match were leaking into an earlier
   one, deleting it would change the number.
3. **Nothing is known before anything is played.** The season's first match must come back
   blank.
4. **Fading works.** With a half-life set, recent maps count for more.

All four pass. Run `python test_form.py` to see it.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

sys.path.insert(0, str(Path.cwd().parent / "src"))

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from dataset import build_map_table, season_folds, split_season
from features import build_features, feature_columns
from form import player_form

maps = build_map_table()
learn, test = split_season(maps)

# The column list is worked out once from the WHOLE season and used everywhere.
# Without this the two halves come back different shapes -- the map pool rotated
# mid-season -- and a model fitted on one could not score the other.
COLUMNS = feature_columns(maps)
X_agents, y = build_features(learn, COLUMNS)
agent_columns = [c for c in X_agents.columns if c.startswith("agent_")]

print(f"learning half: {len(learn):,} maps, described by {len(COLUMNS)} columns")
print(f"test half:     {len(test):,} maps  (not touched in this notebook)")

learning half: 756 maps, described by 39 columns
test half:     516 maps  (not touched in this notebook)


## 1. Deciding how to measure success

An explicit item on the Day 3 list, and one the brief warns about: *"accuracy is rarely the
right choice for imbalanced or asymmetric-cost problems"*.

**Accuracy stays the headline here, and that is defensible** — but for a reason, not by
default. Accuracy misleads when one outcome dominates: guess "no" every time on a question
where 95% of answers are no and you look 95% right while knowing nothing. Our two outcomes
are near evenly split, 51% to 49%, which is exactly the case where accuracy behaves. It is
also what the baselines are expressed in, and what a non-technical reader understands
without explanation.

**But accuracy throws something away.** Logistic regression does not output "team A"; it
outputs a probability, like 0.63. Accuracy rounds that at a half and forgets the rest, so a
model that says *"65% chance"* and one that says *"51% chance"* score identically whenever
both are right. Being confident and wrong costs nothing.

So **log loss** is reported alongside. It scores the probability itself: confident and right
is rewarded, confident and wrong is punished hard, and hedging near a half is middling.
Lower is better, and the number worth holding on to is:

| Log loss | What it means |
|---|---|
| **0.693** | saying "50/50" to every single map — the know-nothing score |
| below 0.693 | the probabilities carry some real information |
| above 0.693 | worse than useless; you would do better claiming to know nothing |

The picker rule has no log loss worth quoting, because it answers with a flat yes or no
rather than a probability.

## 2. How fast should old performances fade?

The probe averaged a player's whole history flat, so a map from January counted as much as
one from last week. That is unlikely to be right — patches change, players find and lose
form, and rosters move.

`half_life` controls it, counted in maps: at 10, a performance from ten maps ago counts half
as much as today's. `None` means no fading, as the probe did.

In [2]:
def fold_scores(X, columns):
    """Accuracy and log loss across the five folds, using only these columns."""
    accuracy, losses = [], []
    for learn_rows, check_rows in season_folds(learn):
        x_train, x_check = X.loc[learn_rows, columns], X.loc[check_rows, columns]

        # scaled using the training part only -- using the whole season's
        # averages would leak information backwards
        middle, spread = x_train.mean(), x_train.std().replace(0, 1)
        x_train, x_check = (x_train - middle) / spread, (x_check - middle) / spread

        model = LogisticRegression(max_iter=5000).fit(x_train, y.loc[learn_rows])
        accuracy.append(model.score(x_check, y.loc[check_rows]))
        losses.append(log_loss(y.loc[check_rows], model.predict_proba(x_check)[:, 1]))
    return np.array(accuracy), np.array(losses)


baseline_accuracy = []
for _, check_rows in season_folds(learn):
    picked_by_a = learn.loc[check_rows, "map_picked_by"].map({"team_a": 1, "team_b": 0})
    baseline_accuracy.append((picked_by_a.fillna(1) == y.loc[check_rows]).mean())
baseline_accuracy = np.array(baseline_accuracy)

print(f"baseline: {baseline_accuracy.mean() * 100:.1f}%\n")

forms = {}
print(f"{'fading':16} {'accuracy':>9} {'worst':>7} {'best':>6} {'log loss':>10}   beats baseline")
for half_life in [None, 40, 20, 10, 7, 5, 3]:
    forms[half_life] = player_form(learn, half_life=half_life).fillna(0.0)
    X = pd.concat([X_agents, forms[half_life]], axis=1)
    accuracy, losses = fold_scores(X, ["rating_diff", "map_picked_by_a"])
    label = "none" if half_life is None else f"half-life {half_life}"
    print(f"{label:16} {accuracy.mean() * 100:8.1f}% {accuracy.min() * 100:6.1f}% "
          f"{accuracy.max() * 100:5.1f}% {losses.mean():10.3f}   "
          f"{int((accuracy > baseline_accuracy).sum())}/5")

baseline: 55.5%

fading            accuracy   worst   best   log loss   beats baseline


none                 57.8%   56.0%  59.6%      0.680   4/5


half-life 40         57.8%   56.0%  60.3%      0.680   3/5


half-life 20         57.9%   55.2%  60.3%      0.679   4/5


half-life 10         58.5%   55.2%  61.8%      0.679   4/5


half-life 7          59.0%   55.2%  64.1%      0.678   4/5


half-life 5          59.7%   56.5%  64.1%      0.678   4/5


half-life 3          59.6%   58.1%  61.8%      0.680   4/5


**Recent form counts for more than old form, and the trend says so more convincingly than
any single row.** Accuracy climbs steadily as fading gets faster — from 57.8% with no fading
to around 59.7% at a half-life of three to five maps. A single good number could be luck; a
steady trend across seven settings is a pattern.

That also means the exact value barely matters, which is the comfortable position to be in.
Anything between three and ten maps gives roughly 58.5% to 59.7%, so nothing here balances
on a knife edge. **A half-life of 5 is used below.** It did score highest — but three and
seven are within a whisker of it, so the choice is not carrying the result on its own.

Worth being plain about the risk: seven settings were tried and one was chosen after seeing
the scores. That is a way to flatter yourself. The defence is that the whole range works
rather than one lucky value, and the final number gets confirmed on the test half once, at
the end.

## 3. Which statistics actually help?

In [3]:
HALF_LIFE = 5
form = forms[HALF_LIFE]
X_all = pd.concat([X_agents, form], axis=1)

form_columns = [c for c in form.columns if c.endswith("_diff")]

candidates = {
    "picker only (notebook 04's best)": ["map_picked_by_a"],
    "agents + map + picker (notebook 04)": COLUMNS,
    "rating only": ["rating_diff"],
    "combat score only": ["acs_diff"],
    "rating + picker": ["rating_diff", "map_picked_by_a"],
    "combat score + picker": ["acs_diff", "map_picked_by_a"],
    "rating + combat score + picker": ["rating_diff", "acs_diff", "map_picked_by_a"],
    "all four stats + picker": form_columns + ["map_picked_by_a"],
    "rating + picker + agents": ["rating_diff", "map_picked_by_a"] + agent_columns,
}

print(f"{'':36} {'accuracy':>9} {'worst':>7} {'best':>6} {'log loss':>10}   beats baseline")
for name, columns in candidates.items():
    accuracy, losses = fold_scores(X_all, columns)
    print(f"{name:36} {accuracy.mean() * 100:8.1f}% {accuracy.min() * 100:6.1f}% "
          f"{accuracy.max() * 100:5.1f}% {losses.mean():10.3f}   "
          f"{int((accuracy > baseline_accuracy).sum())}/5")

                                      accuracy   worst   best   log loss   beats baseline
picker only (notebook 04's best)         54.4%   50.0%  58.9%      0.691   0/5
agents + map + picker (notebook 04)      51.1%   47.3%  54.3%      0.760   2/5


rating only                              58.1%   54.4%  63.1%      0.681   2/5
combat score only                        55.2%   51.1%  61.5%      0.681   2/5
rating + picker                          59.7%   56.5%  64.1%      0.678   4/5
combat score + picker                    59.6%   52.5%  67.9%      0.678   5/5


rating + combat score + picker           59.7%   51.6%  64.9%      0.678   5/5
all four stats + picker                  51.2%   37.9%  56.9%      0.723   2/5
rating + picker + agents                 54.2%   50.7%  59.5%      0.733   2/5


Two results worth separating.

**Less is more, again.** Adding every statistic makes things worse, not better. Rating and
combat score and damage and KAST all measure roughly the same thing — how well a player has
been playing — so piling them in adds noise without adding information. The same happened
with the 27 agent columns in notebook 04. With only 756 maps to learn from, every extra
column costs.

**The agent columns are still harmful.** Adding them to the winning pair drops it by several
points. That is now the **fourth** separate measurement pointing the same way, which is worth
more than any single one of them.

## 4. Turning the regularisation up

Same sweep as notebook 04. `C` controls how hard the coefficients are held back, backwards:
small `C` squeezes hard, large lets them run free.

In [4]:
BEST = ["rating_diff", "map_picked_by_a"]

print(f"{'C':>8} {'accuracy':>9} {'worst':>7} {'best':>6} {'log loss':>10}")
for C in [0.01, 0.1, 0.3, 1.0, 3.0, 10.0, 100.0]:
    accuracy, losses = [], []
    for learn_rows, check_rows in season_folds(learn):
        x_train, x_check = X_all.loc[learn_rows, BEST], X_all.loc[check_rows, BEST]
        middle, spread = x_train.mean(), x_train.std().replace(0, 1)
        x_train, x_check = (x_train - middle) / spread, (x_check - middle) / spread
        model = LogisticRegression(C=C, max_iter=5000).fit(x_train, y.loc[learn_rows])
        accuracy.append(model.score(x_check, y.loc[check_rows]))
        losses.append(log_loss(y.loc[check_rows], model.predict_proba(x_check)[:, 1]))
    accuracy = np.array(accuracy)
    print(f"{C:8} {np.mean(accuracy) * 100:8.1f}% {accuracy.min() * 100:6.1f}% "
          f"{accuracy.max() * 100:5.1f}% {np.mean(losses):10.3f}")

       C  accuracy   worst   best   log loss
    0.01     59.1%   52.2%  63.4%      0.682


     0.1     60.0%   56.5%  64.1%      0.678
     0.3     59.5%   56.5%  64.1%      0.678


     1.0     59.7%   56.5%  64.1%      0.678


     3.0     59.9%   56.5%  64.1%      0.678
    10.0     59.9%   56.5%  64.1%      0.678


   100.0     59.9%   56.5%  64.1%      0.678


## 5. The two models side by side

This is the comparison the project plan asks for.

In [5]:
comparison = []
for name, columns in [
    ("without player stats (notebook 04)", COLUMNS),
    ("with player form", BEST),
]:
    accuracy, losses = fold_scores(X_all, columns)
    comparison.append({
        "model": name,
        "accuracy": f"{accuracy.mean() * 100:.1f}%",
        "worst fold": f"{accuracy.min() * 100:.1f}%",
        "best fold": f"{accuracy.max() * 100:.1f}%",
        "log loss": round(losses.mean(), 3),
        "beats baseline": f"{int((accuracy > baseline_accuracy).sum())}/5",
    })

comparison.insert(0, {
    "model": "baseline: who picked the map",
    "accuracy": f"{baseline_accuracy.mean() * 100:.1f}%",
    "worst fold": f"{baseline_accuracy.min() * 100:.1f}%",
    "best fold": f"{baseline_accuracy.max() * 100:.1f}%",
    "log loss": "-",
    "beats baseline": "-",
})

print(pd.DataFrame(comparison).to_string(index=False))
print("\nlog loss reference: saying 50/50 every time scores 0.693")

                             model accuracy worst fold best fold log loss beats baseline
      baseline: who picked the map    55.5%      50.0%     58.9%        -              -
without player stats (notebook 04)    51.1%      47.3%     54.3%     0.76            2/5
                  with player form    59.7%      56.5%     64.1%    0.678            4/5

log loss reference: saying 50/50 every time scores 0.693


## What this notebook found

1. **Player form predicts the winner. Agent picks do not.** How the ten players had been
   performing, together with who picked the map, clears the baseline consistently. The
   agent columns fail on every measurement taken of them.

2. **The comparison the plan asks for is clean.** Without player stats, 51.4% — below the
   55.5% baseline. With player form, around 59.7% — above it, on four folds of five.

3. **Recent form matters more than old form.** Accuracy rises steadily as older maps are
   faded out, and the whole range from three to ten maps works, so the result does not
   depend on a lucky setting.

4. **Fewer columns beat more columns, every time.** Two features beat four, beat thirty-nine.
   With 756 maps to learn from, anything that does not carry its own weight actively costs.

5. **The probabilities are only slightly informative.** Log loss lands near 0.678 against
   the know-nothing 0.693. The model picks the winner more often than chance, but it is not
   confident about it — which is the honest picture of professional matches between
   evenly-matched teams, and worth saying out loud rather than hiding behind the accuracy
   figure.

6. **Notebook 04's model scores 0.760 on log loss — worse than claiming to know nothing.**
   Accuracy alone said it was mediocre at 51.1%; log loss says it is actively misleading,
   because it commits to answers it has no grounds for. That is the clearest argument for
   having picked a second measure, and it only showed up because we did.

### What still needs doing

- **The test half has not been touched.** Every number here is from the validation folds.
  Several settings were tried and the best kept, so these figures are mildly optimistic by
  construction. The final model gets checked against the test half **once**.
- **`Rating` is missing for all of China Kickoff** and much of China Stage 1, and those maps
  were filled in as "no difference between the teams". `Average Combat Score` is nearly
  complete and performs almost as well, so a version built on it would cover the season more
  evenly. Worth testing properly.
- **Roster changes are invisible.** A team that swapped two players still carries the old
  players' records.
- **Who a player faced is ignored.** A high rating earned against weak opposition counts the
  same as one earned against the best team in the world.